# 01. Credit Card Fraud Detection — Data Quality Assessment & Exploratory Data Analysis (EDA)

**Role Perspective**: Senior Data Analyst / Fraud Risk Specialist  
**Objective**: Perform an audit of 10,000 transaction records, uncover structural fraud anomalies, evaluate financial exposure, and derive actionable risk policies.

---

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## 1. Data Ingestion & Quality Audit
We begin by loading the dataset, verifying column names, checking data types, and testing for null values or duplicate records.

In [ ]:
data_path = os.path.join('..', 'data', 'credit_card_fraud_10k.csv')
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]:,} rows by {df.shape[1]} columns")
df.head()

In [ ]:
# Integrity & completeness verification
null_count = df.isnull().sum().sum()
duplicate_rows = df.duplicated().sum()
duplicate_ids = df['transaction_id'].duplicated().sum()

print(f"Total Null Values: {null_count}")
print(f"Duplicate Rows: {duplicate_rows}")
print(f"Duplicate Transaction IDs: {duplicate_ids}")
assert null_count == 0, "Unexpected nulls found!"
assert duplicate_ids == 0, "Duplicate transaction IDs found!"

## 2. Core Business KPIs & Financial Exposure
> **Critical Risk Management Distinction**: Labeled fraud volume represents gross monetary exposure, not confirmed financial loss. Realized losses are mitigated by pre-authorization decline rules and merchant chargeback clawbacks.

In [ ]:
total_count = len(df)
fraud_count = (df['is_fraud'] == 1).sum()
fraud_rate = (fraud_count / total_count) * 100

total_vol = df['amount'].sum()
fraud_vol = df[df['is_fraud'] == 1]['amount'].sum()
fraud_vol_share = (fraud_vol / total_vol) * 100

print("="*50)
print("PORTFOLIO RISK SCORECARD")
print("="*50)
print(f"Total Transactions Processed: {total_count:,}")
print(f"Fraud-Labeled Transactions:   {fraud_count:,} ({fraud_rate:.2f}%)")
print(f"Total Monetary Volume:        ${total_vol:,.2f}")
print(f"Fraud Monetary Exposure:      ${fraud_vol:,.2f} ({fraud_vol_share:.2f}%)")
print(f"Average Fraud Amount:         ${df[df['is_fraud']==1]['amount'].mean():.2f} (Median: ${df[df['is_fraud']==1]['amount'].median():.2f})")
print(f"Average Legit Amount:         ${df[df['is_fraud']==0]['amount'].mean():.2f} (Median: ${df[df['is_fraud']==0]['amount'].median():.2f})")

## 3. Fraud Pattern Discovery: Geolocation Anomalies
We analyze foreign transactions, billing location mismatches, and their joint interaction.

In [ ]:
# Interaction feature: foreign transaction & location mismatch
df['geo_cohort'] = 'Domestic Match'
df.loc[(df['foreign_transaction'] == 0) & (df['location_mismatch'] == 1), 'geo_cohort'] = 'Domestic Mismatch'
df.loc[(df['foreign_transaction'] == 1) & (df['location_mismatch'] == 0), 'geo_cohort'] = 'Foreign Match'
df.loc[(df['foreign_transaction'] == 1) & (df['location_mismatch'] == 1), 'geo_cohort'] = 'Foreign Mismatch'

geo_summary = df.groupby('geo_cohort').agg(
    total_txns=('is_fraud', 'count'),
    fraud_txns=('is_fraud', 'sum')
).reset_index()
geo_summary['fraud_rate_%'] = (geo_summary['fraud_txns'] / geo_summary['total_txns']) * 100
display(geo_summary)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=geo_summary, x='geo_cohort', y='fraud_rate_%', palette='Reds')
plt.title('Fraud Rate % Across Geolocation Cohorts')
plt.ylabel('Fraud Rate (%)')
plt.xlabel('Cohort')
for i, row in geo_summary.iterrows():
    plt.text(i, row['fraud_rate_%'] + 0.8, f"{row['fraud_rate_%']:.1f}%\n(n={row['total_txns']})", ha='center', fontweight='bold')
plt.ylim(0, 40)
plt.tight_layout()
plt.show()

## 4. Temporal Concentration: The Night Window Effect
Examining transaction hour to identify time-based attack patterns.

In [ ]:
hour_summary = df.groupby('transaction_hour').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
hour_summary['fraud_rate_%'] = (hour_summary['fraud'] / hour_summary['total']) * 100

plt.figure(figsize=(11, 4.5))
plt.plot(hour_summary['transaction_hour'], hour_summary['fraud_rate_%'], marker='o', color='#DC2626', linewidth=2.5)
plt.axvspan(-0.5, 3.5, color='#FEE2E2', alpha=0.6, label='Night Risk Window (00:00 - 04:00)')
plt.title('Hourly Fraud Rate (%) — Extreme Night Spike')
plt.xlabel('Hour of Day (24-Hour Clock)')
plt.ylabel('Fraud Rate (%)')
plt.xticks(range(24))
plt.legend()
plt.tight_layout()
plt.show()

night_frauds = df[df['transaction_hour'].isin([0, 1, 2, 3])]['is_fraud'].sum()
print(f"Frauds in Night Window (00:00-03:59): {night_frauds} out of {fraud_count} total frauds ({night_frauds/fraud_count*100:.1f}%)")

## 5. Device Trust Score & Velocity Surges
Evaluating correlation between machine reputation scores, transaction frequency, and fraud.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Velocity vs Fraud
vel_summary = df.groupby('velocity_last_24h').agg(
    total=('is_fraud', 'count'),
    fraud=('is_fraud', 'sum')
).reset_index()
vel_summary['fraud_rate_%'] = (vel_summary['fraud'] / vel_summary['total']) * 100
sns.barplot(ax=axes[0], data=vel_summary, x='velocity_last_24h', y='fraud_rate_%', color='#2563EB')
axes[0].set_title('Fraud Rate % by 24-Hour Velocity')
axes[0].set_ylabel('Fraud Rate (%)')

# Device trust score distribution
sns.kdeplot(ax=axes[1], data=df[df['is_fraud']==0]['device_trust_score'], label='Legitimate (0)', fill=True, color='#3B82F6')
sns.kdeplot(ax=axes[1], data=df[df['is_fraud']==1]['device_trust_score'], label='Fraud (1)', fill=True, color='#DC2626')
axes[1].set_title('Device Trust Score KDE: Legit vs Fraud')
axes[1].set_xlabel('Device Trust Score (0-100)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Statistical Hypothesis Testing
We perform Chi-Squared contingency tests on key categorical drivers to ensure observed associations are not random sampling artifacts.

In [ ]:
for col in ['foreign_transaction', 'location_mismatch', 'merchant_category']:
    contingency = pd.crosstab(df[col], df['is_fraud'])
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    sig = "Statistically Significant (p < 0.001)" if p < 0.001 else "Not Significant"
    print(f"Feature: {col:<22} | Chi2: {chi2:>8.2f} | p-value: {p:.2e} | {sig}")